# CIFAR-10 MobileNetV2: pruning + KD recovery

Attach a Kaggle dataset containing this repository and the existing `results/checkpoints/baseline.pt` plus selected QAT resume checkpoint. This notebook writes every result beneath `/kaggle/working/assignment-2/experiments/`; the final cell makes a downloadable zip. It does not claim sparse inference speedup.

In [ ]:
from pathlib import Path
import os, shutil, subprocess

# Change this only to the attached Kaggle dataset slug/directory.
INPUT_ROOT = Path('/kaggle/input/assignment-2')
WORKDIR = Path('/kaggle/working/assignment-2')
if WORKDIR.exists(): shutil.rmtree(WORKDIR)
shutil.copytree(INPUT_ROOT, WORKDIR)
os.chdir(WORKDIR)
print(WORKDIR)
!python -m pip install -q -r requirements.txt
!nvidia-smi

In [ ]:
# Required correctness/packing gates before a GPU run.
!python -m unittest discover -s tests -v
!sha256sum results/checkpoints/baseline.pt
# Set this to the selected *resume checkpoint*, never the .qpk artifact.
QAT_CHECKPOINT = 'results/checkpoints/qat-mp-w4dw6edgew8-a6-seed6886-best-target.pt'
assert Path(QAT_CHECKPOINT).exists(), QAT_CHECKPOINT

In [ ]:
# Zero-shot 30/40/50% masks; recover the two validation-best candidates with frozen BN and KD.
# Add --evaluate-test only after choosing the validation candidate (normally a separate final run).
!python -m src.prune_qat_kd --checkpoint {QAT_CHECKPOINT} --teacher-checkpoint results/checkpoints/baseline.pt --device cuda --epochs 4 --sparsities 0.30 0.40 0.50 --max-recoveries 2 --run-name kaggle-pruned-mp-w4dw6edgew8-a6-seed6886 | tee pruning_kd.log
!find experiments/pruning_kd -maxdepth 3 -type f | sort

In [ ]:
import json
result_file = WORKDIR / 'experiments/pruning_kd/kaggle-pruned-mp-w4dw6edgew8-a6-seed6886/results.json'
results = json.loads(result_file.read_text())
winner = results['validation_selected']
print(json.dumps(winner, indent=2))
print('Compression ratio is exact serialized QPK2 bytes. Activation accounting remains separate; no latency conclusion follows.')

In [ ]:
# Optional longer-horizon dense alternative from DISTILLATION_QAT.md.
# !python -m src.distill --stage fp32 --teacher-checkpoint results/checkpoints/baseline.pt --width-mult 0.75 --epochs 30 --device cuda --run-name mobilenetv2-075-fp32-kd
# !python -m src.distill --stage qat --teacher-checkpoint results/checkpoints/baseline.pt --student-checkpoint experiments/student_distillation/mobilenetv2-075-fp32-kd/best.pt --width-mult 0.75 --epochs 10 --device cuda --run-name mobilenetv2-075-qat-kd

In [ ]:
archive = shutil.make_archive('/kaggle/working/pruning_distillation_results', 'zip', WORKDIR / 'experiments')
print('Download:', archive)
# Kaggle Output will persist the zip when you Save Version.